# Pandas Lesson 4: Group By and Aggregating

**Group by** puts together the rows that share the same value in a column, so we can treat them as one group.

**Aggregating** means calculating one number per group: an average, a count, a total, a minimum, a maximum...

Together, they answer questions like: *"On average, do chocolate-based ice creams get better ratings than vanilla-based ones?"*

### What you will learn

1. Grouping rows with `groupby()`
2. Basic aggregations: `mean`, `count`, `min`, `max`, `sum`
3. Several aggregations at once with `agg()`
4. Grouping by several columns
5. A quick overview with `describe()`

> This notebook follows the pandas tutorial series by Alex The Analyst, with my own notes and explanations.

## 0. Setup

### The dataset

This time we use a small, simple dataset: `Flavors.csv`. It contains 9 ice cream flavors:

| Column | Meaning |
|---|---|
| `Flavor` | the name of the ice cream |
| `Base Flavor` | its base: `Chocolate` or `Vanilla` |
| `Liked` | whether it was liked: `Yes` or `No` |
| `Flavor Rating` | a score for the taste, out of 10 |
| `Texture Rating` | a score for the texture, out of 10 |
| `Total Rating` | Flavor Rating + Texture Rating |

The ratings are Alex's personal (and very serious) opinion. A small dataset like this makes it easy to check every result by hand.

In [1]:
import pandas as pd

df = pd.read_csv('data/Flavors.csv')
df

,Flavor,Base Flavor,Liked,Flavor Rating,Texture Rating,Total Rating
0,Mint Chocolate Chip,Vanilla,Yes,10.0,8.0,18.0
1,Chocolate,Chocolate,Yes,8.8,7.6,16.6
2,Vanilla,Vanilla,No,4.7,5.0,9.7
3,Cookie Dough,Vanilla,Yes,6.9,6.5,13.4
4,Rocky Road,Chocolate,Yes,8.2,7.0,15.2
5,Pistachio,Vanilla,No,2.3,3.4,5.7
6,Cake Batter,Vanilla,Yes,6.5,6.0,12.5
7,Neapolitan,Vanilla,No,3.8,5.0,8.8
8,Chocolte Fudge Brownie,Chocolate,Yes,8.2,7.1,15.3


---
## 1. Grouping Rows with `groupby()`

### Choosing the right column

To group, we need a column whose values **repeat**.

- `Flavor` is a bad choice: every ice cream is different, so each group would contain a single row. The average of one rating is just that rating, so it tells us nothing.
- `Base Flavor` is a good choice: it only has two values (`Chocolate` and `Vanilla`) that repeat, so we get two groups we can compare.

### The GroupBy object

Let's group by `Base Flavor` and see what we get:

In [2]:
df.groupby('Base Flavor')

We don't get a table, but a **`DataFrameGroupBy` object**. The rows have been split into groups, but pandas doesn't know yet **what to calculate** for each group. An aggregation is still missing.

We store the groups in a variable so we can reuse them:

In [3]:
group_by_frame = df.groupby('Base Flavor')

---
## 2. Basic Aggregations

### `mean()`: the average

`mean()` calculates the average of each column, for each group.

`numeric_only=True` tells pandas to keep only the columns with **numbers**. Without it, pandas would also try to average the text columns (`Flavor`, `Liked`), and that raises an error, because you can't calculate the average of words.

> In Alex's video, `mean()` skips the text columns automatically. Newer versions of pandas don't do that anymore, so we add `numeric_only=True`.

In [4]:
group_by_frame.mean(numeric_only=True)

,Flavor Rating,Texture Rating,Total Rating
Base Flavor,,,
Chocolate,8.4,7.233333,15.70
Vanilla,5.7,5.650000,11.35


Notice that `Base Flavor` is now the **index** of the result (the bold labels on the left), with one row per group.

At first glance, chocolate-based ice creams have much better average ratings than vanilla-based ones.

You can also write everything **in one line**, without the intermediate variable. This is how you will see it most often:

In [5]:
df.groupby('Base Flavor').mean(numeric_only=True)

,Flavor Rating,Texture Rating,Total Rating
Base Flavor,,,
Chocolate,8.4,7.233333,15.70
Vanilla,5.7,5.650000,11.35


### `count()`: the number of rows

`count()` tells how many rows are in each group.

In [6]:
group_by_frame.count()

,Flavor,Liked,Flavor Rating,Texture Rating,Total Rating
Base Flavor,,,,,
Chocolate,3,3,3,3,3
Vanilla,6,6,6,6,6


There are only **3** chocolate ice creams, against **6** vanilla ones. This is important context for the averages above: with only 3 rows, a couple of very good chocolate flavors can pull the average up a lot. Always check the count before trusting an average.

### `min()` and `max()`

`min()` returns the smallest value of each column, and `max()` the largest.

In [7]:
group_by_frame.min()

,Flavor,Liked,Flavor Rating,Texture Rating,Total Rating
Base Flavor,,,,,
Chocolate,Chocolate,Yes,8.2,7.0,15.2
Vanilla,Cake Batter,No,2.3,3.4,5.7


This time the text columns (`Flavor`, `Liked`) are included. For text, "smallest" means **first in alphabetical order**: for vanilla, `Cake Batter` comes first, and `No` comes before `Yes`.

For chocolate, the minimum of `Liked` is `Yes`: that means **every** chocolate ice cream was liked.

In [8]:
group_by_frame.max()

,Flavor,Liked,Flavor Rating,Texture Rating,Total Rating
Base Flavor,,,,,
Chocolate,Rocky Road,Yes,8.8,7.6,16.6
Vanilla,Vanilla,Yes,10.0,8.0,18.0


`max()` does the opposite: the last value in alphabetical order for text (`Rocky Road`, `Vanilla`), and the highest number for the ratings.

The best rated ice cream overall is vanilla-based: one vanilla flavor got a **10** for taste (it's Mint Chocolate Chip).

### `sum()`: the total

`sum()` adds up the values of each group:

In [9]:
group_by_frame.sum()

,Flavor,Liked,Flavor Rating,Texture Rating,Total Rating
Base Flavor,,,,,
Chocolate,ChocolateRocky RoadChocolte Fudge Brownie,YesYesYes,25.2,21.7,47.1
Vanilla,Mint Chocolate ChipVanillaCookie DoughPistachi...,YesNoYesNoYesNo,34.2,33.9,68.1


Look at the text columns: for text, "adding" means **sticking the words together**, which gives strange results like `YesYesYes`. That's not useful, so we add `numeric_only=True` to keep only the number columns:

In [10]:
group_by_frame.sum(numeric_only=True)

,Flavor Rating,Texture Rating,Total Rating
Base Flavor,,,
Chocolate,25.2,21.7,47.1
Vanilla,34.2,33.9,68.1


Vanilla has a higher total than chocolate, but only because it has **twice as many rows**. That's why totals must always be read together with the count.

---
## 3. Several Aggregations at Once with `agg()`

Calling `mean()`, `max()`, `count()`, `sum()` one by one is repetitive. `agg()` (short for *aggregate*) lets us run them **all in one go**.

We give `agg()` a **dictionary**:

- the **key** is the column to calculate on,
- the **value** is a list of the calculations we want.

```python
{'column name': ['calculation 1', 'calculation 2', ...]}
```

### On one column

In [11]:
df.groupby('Base Flavor').agg({'Flavor Rating': ['mean', 'max', 'count', 'sum']})

Flavor Rating                  
                     mean   max count   sum
Base Flavor                                
Chocolate             8.4   8.8     3  25.2
Vanilla               5.7  10.0     6  34.2

We get one column (`Flavor Rating`) split into several sub-columns, one per calculation.

We don't need `numeric_only=True` here, because we chose only one column, and it contains numbers.

### On several columns

We just add another `key: value` pair to the dictionary, separated by a comma:

In [12]:
df.groupby('Base Flavor').agg({
    'Flavor Rating': ['mean', 'max', 'count', 'sum'],
    'Texture Rating': ['mean', 'max', 'count', 'sum']
})

Flavor Rating                   Texture Rating                 
                     mean   max count   sum           mean  max count   sum
Base Flavor                                                                
Chocolate             8.4   8.8     3  25.2       7.233333  7.6     3  21.7
Vanilla               5.7  10.0     6  34.2       5.650000  8.0     6  33.9

---
## 4. Grouping by Several Columns

So far we grouped on `Base Flavor` only. We can also group on **several columns** by giving `groupby()` a **list**.

Here we group by `Base Flavor`, then by `Liked` inside each base:

In [13]:
df.groupby(['Base Flavor', 'Liked']).mean(numeric_only=True)

Flavor Rating  Texture Rating  Total Rating
Base Flavor Liked                                             
Chocolate   Yes              8.4        7.233333     15.700000
Vanilla     No               3.6        4.466667      8.066667
            Yes              7.8        6.833333     14.633333

The result has a **multi-index** (two levels on the left), like the ones we built in Lesson 3.

- **Chocolate** only has a `Yes` row, since every chocolate ice cream was liked.
- **Vanilla** has both. The vanilla ice creams that were liked score **almost as high** as the chocolate ones, while the ones that weren't liked score very low.

So the low vanilla average from section 2 comes from a few bad flavors, not from vanilla in general. Grouping on a second column revealed that.

`agg()` works the same way with several groups:

In [14]:
df.groupby(['Base Flavor', 'Liked']).agg({'Flavor Rating': ['mean', 'max', 'count', 'sum']})

Flavor Rating                  
                           mean   max count   sum
Base Flavor Liked                                
Chocolate   Yes             8.4   8.8     3  25.2
Vanilla     No              3.6   4.7     3  10.8
            Yes             7.8  10.0     3  23.4

---
## 5. A Quick Overview with `describe()`

`describe()` is a shortcut that calculates many statistics at once for every number column:

| Statistic | Meaning |
|---|---|
| `count` | number of values |
| `mean` | average |
| `std` | standard deviation: how spread out the values are |
| `min` / `max` | smallest / largest value |
| `25%`, `50%`, `75%` | percentiles: 25% of the values are below the `25%` value, and so on (`50%` is the median, the middle value) |

In [15]:
df.groupby('Base Flavor').describe()

Flavor Rating                                             \
                    count mean       std  min    25%  50%  75%   max   
Base Flavor                                                            
Chocolate             3.0  8.4  0.346410  8.2  8.200  8.2  8.5   8.8   
Vanilla               6.0  5.7  2.710719  2.3  4.025  5.6  6.8  10.0   

            Texture Rating            ...             Total Rating         \
                     count      mean  ...    75%  max        count   mean   
Base Flavor                           ...                                   
Chocolate              3.0  7.233333  ...  7.350  7.6          3.0  15.70   
Vanilla                6.0  5.650000  ...  6.375  8.0          6.0  11.35   

                                                         
                  std   min     25%   50%     75%   max  
Base Flavor                                              
Chocolate    0.781025  15.2  15.250  15.3  15.950  16.6  
Vanilla      4.263684   5.7   9.025  11.1  13.175  18.0  

[2 rows x 24 columns]

`describe()` is great for a fast first look, but you can't choose the calculations. When you need specific ones, use `agg()`.

> The `std` confirms what we saw: chocolate ratings are very close to each other (small `std`), while vanilla ratings are much more spread out (large `std`).

---
## Summary

| Goal | Code |
|---|---|
| Group rows by a column | `df.groupby('Base Flavor')` |
| Average per group | `df.groupby('Base Flavor').mean(numeric_only=True)` |
| Number of rows per group | `df.groupby('Base Flavor').count()` |
| Smallest / largest value | `.min()` / `.max()` |
| Total per group | `.sum(numeric_only=True)` |
| Several calculations on one column | `.agg({'Flavor Rating': ['mean', 'max', 'count', 'sum']})` |
| Several calculations on several columns | `.agg({'col1': [...], 'col2': [...]})` |
| Group by several columns | `df.groupby(['Base Flavor', 'Liked'])` |
| Quick statistical overview | `df.groupby('Base Flavor').describe()` |

**Next lesson:** Merging, Joining and Concatenating.